# Experiment 3: Sobol quasi-random search

This notebook runs 32 well-spread graph settings for no-PCA, PCA-50, and PCA-100. It loads a validated core run and a separately selected PCA fit.


In [ ]:
from pathlib import Path
import sys


def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")


PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")


In [ ]:
from datetime import datetime
import anndata as ad
import pandas as pd

from cross_species import load_core_run
from cross_species.pca import load_nested_pca_scores, run_sobol_pca_search


In [ ]:
from cross_species import load_core_run

CORE_RUN_DIR = None  # Set explicitly, or leave None to use the newest core_* run.
if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No core run found. Run 01_full_pipeline.ipynb first.")
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
graph = core["graph"]
labels = core["labels"]
display(pd.Series(core["manifest"], name="value"))


## Inputs and representation-specific ranges


In [ ]:
PCA_ARTIFACT_DIR = None  # Point to results/pca/pca_fit_*.
PARAMETER_RANGES = {
    "no_pca": {
        "k_intra": (10, 500),
        "k_inter": (10, 500),
        "cutoff_intra": (None, None),
        "cutoff_inter": (None, None),
    },
    "pca_050": {
        "k_intra": (10, 500),
        "k_inter": (10, 500),
        "cutoff_intra": (None, None),
        "cutoff_inter": (None, None),
    },
    "pca_100": {
        "k_intra": (10, 500),
        "k_inter": (10, 500),
        "cutoff_intra": (None, None),
        "cutoff_inter": (None, None),
    },
}
pd.DataFrame(PARAMETER_RANGES).T


## Run the search


In [ ]:
RUN_SOBOL_SEARCH = False
if RUN_SOBOL_SEARCH:
    if PCA_ARTIFACT_DIR is None:
        raise ValueError("Set PCA_ARTIFACT_DIR")
    pca_scores, _, score_path = load_nested_pca_scores(
        PCA_ARTIFACT_DIR, minimum_dimensions=100
    )
    sm_adata_human = ad.read_h5ad(
        DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
    )
    sm_adata_mouse = ad.read_h5ad(
        DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"
    )
    output_dir = RESULTS_ROOT / "sobol" / datetime.now().strftime(
        "sobol_pca_pcc_%Y%m%d_%H%M%S"
    )
    sobol = run_sobol_pca_search(
        prof,
        C,
        pca_scores=pca_scores,
        parameter_ranges_by_representation=PARAMETER_RANGES,
        expression_by_species={"mouse": sm_adata_mouse, "human": sm_adata_human},
        n_settings=32,
        batch_size=4,
        output_dir=output_dir,
        show_summary_plots=True,
    )
    display(sobol["summary"])
else:
    print("Enter every cutoff range, set PCA_ARTIFACT_DIR, then enable RUN_SOBOL_SEARCH.")
